In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_score

1. Загрузите выборку Boston с помощьюфункцииsklearn.datasets.load_boston().
Результатом вызова данной функции является объект, у которого
признаки записаны в поле data, а целевой вектор в поле target.

In [6]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
pd.DataFrame(data).head()
#pd.DataFrame(target).head()

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Ярослав\AppData\Local\Temp\ipykernel_17264\847070280.py:2: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33


In [10]:
print(f"Тип data: {type(data)}")
print(f"Форма data: {data.shape}")
print(f"Тип target: {type(target)}")
print(f"Форма target: {target.shape}")
print(f"Первые 5 значений target: {target[:5]}")
print(f"Первые 5 строк data (первые 5 признаков):\n{pd.DataFrame(data[:5, :5])}")
print(f"Пропуски в data: {np.isnan(data).sum()}")
print(f"Пропуски в target: {np.isnan(target).sum()}")

Тип data: <class 'numpy.ndarray'>
Форма data: (506, 13)
Тип target: <class 'numpy.ndarray'>
Форма target: (506,)
Первые 5 значений target: [24.  21.6 34.7 33.4 36.2]
Первые 5 строк data (первые 5 признаков):
          0         1         2         3         4
0 -0.419782  0.284830 -1.287909 -0.272599 -0.144217
1 -0.417339 -0.487722 -0.593381 -0.272599 -0.740262
2 -0.417342 -0.487722 -0.593381 -0.272599 -0.740262
3 -0.416750 -0.487722 -1.306878 -0.272599 -0.835284
4 -0.412482 -0.487722 -1.306878 -0.272599 -0.835284
Пропуски в data: 0
Пропуски в target: 0


2. Приведите признаки в выборке к одному масштабу при помощи
функции sklearn.preprocessing.scale.
3. Переберите разные варианты параметра метрики p по сетке от 1 до
10 с таким шагом, чтобы всего было протестировано 200 вариантов
(используйте функцию numpy.linspace). Используйте KNeighborsRegressor
с n_neighbors=5 и weights=’distance’ данный параметр добавляет
в алгоритм веса, зависящие от расстояния до ближайших соседей. В
качестве метрики качества используйте среднеквадратичную ошиб
ку (параметр scoring=’mean_squared_error’ у cross_val_score; при
использовании библиотеки scikit-learn версии 18.0.1 и выше необхо
димо указывать scoring=’neg_mean_squared_error’). Качество оце
нивайте, как и в предыдущем задании, с помощью кросс-валидации
по 5 блокам с random_state = 42, не забудьте включить перемеши
вание выборки (shuffle=True).
4. Определите, при каком p качество на кросс-валидации оказалось
оптимальным. Обратите внимание, что cross_val_score возвращает массив показателей качества по блокам; необходимо максимизи
ровать среднее этих показателей. Это значение параметра и будет
ответом на задачу.

In [12]:
data = scale(data)
kfold = KFold(n_splits = 5, shuffle = True, random_state=42)
x = np.linspace(1, 10, 200)
mse_scores = []
for i, p in enumerate(x):
    knn = KNeighborsRegressor(n_neighbors=5, weights = 'distance', metric = 'minkowski', p = p)
    scores = cross_val_score(knn, data, target, cv = kfold, scoring = 'neg_mean_squared_error')
    mean_score = scores.mean()
    mse_scores.append(mean_score)
    if i % 20 == 0:  # выводим каждое 20-е значение
        print(f"p={p:.4f}, MSE={-mean_score:.4f}")

best_idx = np.argmax(mse_scores)
best_p = x[best_idx]
best_mse = -mse_scores[best_idx]
print("Оптимальное p: ", best_p)
print("Среднеквадратичная ошибка: ", best_mse)

p=1.0000, MSE=16.0306
p=1.9045, MSE=17.4151
p=2.8090, MSE=17.6147
p=3.7136, MSE=18.9105
p=4.6181, MSE=19.4686
p=5.5226, MSE=19.8849
p=6.4271, MSE=20.5587
p=7.3317, MSE=20.9937
p=8.2362, MSE=20.8927
p=9.1407, MSE=21.0775
Оптимальное p:  1.0
Среднеквадратичная ошибка:  16.030646734221644
